# Домашнее задание 2. Своя среда: управление запасами

**Вес в оценке:** базовое ДЗ, среднее по сложности.

Задание состоит из трёх частей:

1. **Практика: среда `InventoryEnv` как `gym.Env` (40 баллов)**
2. **Практика: две версии награды и сравнение обучения (35 баллов)**
3. **Теория: среда как MDP (25 баллов)**

Части с `assert` проверяются автоматически при запуске ячейки — если assert не упал, эта часть засчитана. Текстовые ответы и графики проверяются вручную.

## Задача

Вы управляете складом одного товара. Каждый день:

1. утром на складе `stock` единиц (от 0 до `capacity`);
2. вы решаете, сколько единиц **заказать**: действие $a \in \{0, 1, \ldots, \text{max\_order}\}$; заказ приезжает сразу, но склад не может вместить больше `capacity` — лишнее пропадает;
3. днём приходит **случайный спрос** $d$ (например, Пуассон со средним `demand_mean`), продаётся $\min(d, \text{stock})$ единиц;
4. вечером остаток переходит на завтра. Эпизод длится `horizon` дней.

Экономика дня: выручка `price` за проданную единицу, `order_cost` за заказанную, `holding_cost` за каждую единицу, оставшуюся на складе к вечеру, и `stockout_cost` за каждую единицу неудовлетворённого спроса.

Это классическая задача исследования операций, и RL здесь — не единственный способ, зато прекрасный полигон: дискретное состояние, дискретное действие, честная случайность и награда с несколькими слагаемыми, которые легко разбалансировать.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env

## Часть 1. Среда `InventoryEnv` (40 баллов)

### 1.1 Класс среды (25 баллов)

Реализуйте `InventoryEnv`:

* `observation_space = Discrete(capacity + 1)` — наблюдение = текущий остаток;
* `action_space = Discrete(max_order + 1)`;
* `reset(seed)` — остаток `initial_stock`, день 0, вернуть `(obs, info)`;
* `step(action)` — по описанию выше; награда за день = выручка − стоимость заказа − хранение − штраф за дефицит; `terminated=True` после `horizon`-го дня; в `info` положите `demand` и `sold`;
* спрос генерируйте через `self.np_random.poisson(self.demand_mean)` — тогда среда воспроизводима по сиду.

Подумайте, какой из флагов — `terminated` или `truncated` — правильно выставлять в конце горизонта, и объясните выбор в комментарии. (Подсказка: после `horizon` дней задача **закончена по условию**, а не оборвана снаружи.)

In [ ]:
class InventoryEnv(gym.Env):
    metadata = {"render_modes": ["ansi"]}

    def __init__(self, capacity=20, max_order=10, demand_mean=4.0, horizon=30, initial_stock=10,
                 price=5.0, order_cost=2.0, holding_cost=0.5, stockout_cost=3.0, render_mode=None):
        self.capacity, self.max_order = capacity, max_order
        self.demand_mean, self.horizon, self.initial_stock = demand_mean, horizon, initial_stock
        self.price, self.order_cost = price, order_cost
        self.holding_cost, self.stockout_cost = holding_cost, stockout_cost
        self.render_mode = render_mode

        # TODO: observation_space и action_space
        self.observation_space = None
        self.action_space = None

        self.stock = initial_stock
        self.day = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # TODO
        raise NotImplementedError

    def step(self, action):
        # TODO: заказ -> спрос -> продажа -> награда -> конец горизонта
        raise NotImplementedError

    def render(self):
        if self.render_mode == "ansi":
            return f"день {self.day:2d}: на складе {self.stock:2d}"

In [ ]:
# Самопроверка 1.1
env = InventoryEnv()
check_env(env)

obs, info = env.reset(seed=0)
assert obs == 10 and env.observation_space.contains(obs)
assert env.action_space.n == 11 and env.observation_space.n == 21

# заказ выше вместимости обрезается: 10 + 10 = 20 -> не больше capacity
env.reset(seed=0)
obs, r, term, trunc, info = env.step(10)
assert 0 <= obs <= 20 and "demand" in info and "sold" in info
assert info["sold"] <= 20 and obs == 20 - info["sold"], "продано не может быть больше остатка после заказа"

# эпизод длится ровно horizon дней
env.reset(seed=1)
for day in range(30):
    obs, r, term, trunc, info = env.step(0)
    assert term == (day == 29), "terminated должен стать True ровно после horizon-го дня"

# воспроизводимость: одинаковый seed -> одинаковый спрос
def demands(seed):
    e = InventoryEnv(); e.reset(seed=seed)
    return [e.step(3)[4]["demand"] for _ in range(10)]
assert demands(42) == demands(42) and demands(42) != demands(43)
print("OK: InventoryEnv проходит проверки")

### 1.2 Базовые линии (15 баллов)

Реализуйте функцию `evaluate(env_factory, policy, n_episodes, seed)` — средний return политики `policy(obs) -> action` по `n_episodes` эпизодам — и сравните три ручные политики:

1. **ничего не заказывать**;
2. **заказывать фиксированно** `k` единиц каждый день (подберите `k` перебором от 0 до 10);
3. **пополнять до уровня** $S$: заказать $\max(0, S - \text{stock})$ (классическая «base-stock policy»; подберите $S$).

Постройте график «средний return в зависимости от параметра» для политик 2 и 3. Какая из ручных политик лучшая и почему это ожидаемо?

In [ ]:
def evaluate(env_factory, policy, n_episodes=100, seed=0):
    # TODO: средний return по n_episodes эпизодам; сиды эпизодов seed, seed+1, ...
    raise NotImplementedError

# TODO: три политики, перебор параметров, график

In [ ]:
# Самопроверка 1.2
_r = evaluate(InventoryEnv, lambda obs: 0, n_episodes=20)
assert isinstance(_r, float) or isinstance(_r, np.floating)
print(f"OK: evaluate работает; return политики «ничего не заказывать» = {_r:.1f}")

## Часть 2. Две версии награды (35 баллов)

### 2.1 Cross-Entropy на среде (10 баллов)

Ниже — табличный Cross-Entropy из лекции. Обучите его на `InventoryEnv` (состояний 21, действий 11) и сравните с лучшей ручной политикой из 1.2. Постройте кривую обучения и **изобразите выученную политику**: график «остаток на складе → самое вероятное количество заказа». Похожа ли она на «пополнять до уровня»?

Подсказка: у нас return — сумма за 30 дней с большим разбросом, квантиль $q = 0.5$–$0.7$ и сглаживание `laplace=1, mix=0.5` работают надёжнее, чем жёсткие обновления.

Не удивляйтесь, если Cross-Entropy **проиграет** лучшей ручной политике из 1.2 — за 30 итераций так обычно и бывает. Объясните текстом, почему: сколько параметров у таблицы $21 \times 11$, сколько эпизодов реально «видит» каждую клетку, и почему стохастическая политика в задаче с чётким правилом «пополнять до уровня» заведомо теряет. Что можно сделать (больше итераций, другой квантиль, детерминизация после обучения)? Попробуйте одно улучшение и покажите результат.

In [ ]:
def run_session(env, policy, rng, max_steps=100):
    """Один эпизод политикой-таблицей: состояния, действия, суммарная награда."""
    obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
    states, actions, total = [], [], 0.0
    for _ in range(max_steps):
        a = int(rng.choice(policy.shape[1], p=policy[obs]))
        states.append(obs); actions.append(a)
        obs, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    return states, actions, total


def cross_entropy_method(env, n_iter=25, n_sessions=200, q=0.7, laplace=0.5, mix=0.5,
                         seed=0, max_steps=100, evaluate=None):
    """Табличный Cross-Entropy из лекции 1. evaluate(policy) -> число, если хотим отдельную метрику."""
    rng = np.random.default_rng(seed)
    n_states, n_actions = env.observation_space.n, env.action_space.n
    policy = np.ones((n_states, n_actions)) / n_actions
    log = {"mean": [], "eval": []}
    for it in range(n_iter):
        sessions = [run_session(env, policy, rng, max_steps) for _ in range(n_sessions)]
        returns = np.array([G for _, _, G in sessions])
        threshold = np.quantile(returns, q)
        elite = [s for s in sessions if s[2] >= threshold and s[2] > returns.min()]
        counts = np.full((n_states, n_actions), laplace)
        for states, actions, _ in elite:
            for s, a in zip(states, actions):
                counts[s, a] += 1
        new_policy = policy.copy()
        seen = counts.sum(axis=1) > 0
        new_policy[seen] = counts[seen] / counts[seen].sum(axis=1, keepdims=True)
        policy = mix * new_policy + (1 - mix) * policy
        log["mean"].append(returns.mean())
        if evaluate is not None:
            log["eval"].append(evaluate(policy))
    return policy, log


# TODO: обучение, кривая, картинка политики

### 2.2 Разреженная награда (15 баллов)

Сделайте обёртку `SparseReward(gym.Wrapper)`, которая накапливает настоящую награду внутри эпизода, но **выдаёт её агенту одним числом в последний день**, а в остальные дни возвращает 0. Это та же задача с той же оптимальной политикой — меняется только *когда* агент узнаёт результат.

Обучите Cross-Entropy на обёрнутой среде с теми же гиперпараметрами. Оценивайте обе политики (из 2.1 и 2.2) **одной и той же** функцией `evaluate` на исходной среде. Сравните кривые обучения и итоговые returns. Изменилось ли что-то — и почему для Cross-Entropy это (не)важно? Ответьте текстом: какой класс алгоритмов заметил бы разницу между этими двумя средами (вспомните, на чём учится Cross-Entropy, а на чём будут учиться методы недели 3).

In [ ]:
class SparseReward(gym.Wrapper):
    def reset(self, seed=None, options=None):
        # TODO: обнулить накопитель
        raise NotImplementedError

    def step(self, action):
        # TODO: накапливать награду, выдавать сумму только когда terminated или truncated
        raise NotImplementedError

# TODO: обучение на SparseReward(InventoryEnv()), сравнение с 2.1

In [ ]:
# Самопроверка 2.2: суммарная награда за эпизод не меняется, меняется только её распределение по шагам.
_e_dense, _e_sparse = InventoryEnv(), SparseReward(InventoryEnv())
_e_dense.reset(seed=3); _e_sparse.reset(seed=3)
_dense_rewards, _sparse_rewards = [], []
for _ in range(30):
    _dense_rewards.append(_e_dense.step(4)[1]); _sparse_rewards.append(_e_sparse.step(4)[1])
assert all(r == 0 for r in _sparse_rewards[:-1]), "до последнего дня награда должна быть 0"
assert abs(sum(_dense_rewards) - sum(_sparse_rewards)) < 1e-6, "суммы за эпизод должны совпадать"
print("OK: SparseReward сохраняет суммарную награду")

### 2.3 Сломанная награда (10 баллов)

Придумайте **правдоподобную** модификацию награды, которая выглядит разумно, но приводит к неправильному поведению агента (например: бонус за «полный склад», штраф только за дефицит без стоимости заказа, положительная награда за каждый день работы). Обучите Cross-Entropy на ней, оцените результат по **настоящей** награде и опишите, что именно агент научился эксплуатировать. Одна такая история — 10 баллов.

In [ ]:
# TODO: обёртка с «сломанной» наградой, обучение, честная оценка, вывод

## Часть 3. Теория: среда как MDP (25 баллов)

Ответы пишите в ячейке ниже (Markdown, можно с формулами).

### 3.1 Формальное описание (10 баллов)

Опишите `InventoryEnv` как MDP $\langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle$: явно выпишите множества $\mathcal{S}$ и $\mathcal{A}$, формулу для $P(s' \mid s, a)$ через распределение спроса (не забудьте про обрезание по `capacity` и про то, что $s' = 0$ достигается многими значениями спроса), формулу для $R(s, a)$ как ожидаемой награды за день. Какое $\gamma$ разумно взять для горизонта в 30 дней и почему?

### 3.2 Марковость (5 баллов)

Является ли наблюдение (текущий остаток) полным состоянием? Что изменится, если спрос станет зависеть от дня недели, а день в наблюдение не входит? Как это починить, не меняя динамику среды?

### 3.3 Terminated или truncated (5 баллов)

Вы выбрали один из флагов в конце горизонта. Объясните, к чему привело бы противоположное решение для алгоритма, который считает ценность состояния как «награда + $\gamma$ · ценность следующего состояния» (неделя 3).

### 3.4 Shaping (5 баллов)

Предложите потенциальную функцию $\Phi(s)$ для этой задачи и выпишите слагаемое $F(s, s') = \gamma\Phi(s') - \Phi(s)$. Почему такой shaping не изменит оптимальную политику, а бонус «+1 за каждый день с ненулевым остатком» — может?

*Ваши ответы на часть 3:*

**3.1**

**3.2**

**3.3**

**3.4**

## Чек-лист перед сдачей

- [ ] `InventoryEnv` проходит `check_env` и самопроверку 1.1
- [ ] Три ручные политики оценены, есть график по параметру, выбрана лучшая
- [ ] Cross-Entropy обучен, есть кривая обучения и картинка выученной политики
- [ ] `SparseReward` проходит самопроверку, есть сравнение с плотной наградой и текстовый ответ
- [ ] Есть история про сломанную награду с честной оценкой
- [ ] Часть 3: MDP выписан формально, ответы на 3.2–3.4 даны